<a href="https://colab.research.google.com/github/Jannatu37/cgm-project/blob/main/Guessing_game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — install libraries
!pip install -q tensorflow==2.20.0 tensorflow-datasets gradio==3.35.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.5/84.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 101.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.20.0 which is incompatible.


In [ ]:
# Cell 2 — imports & helper functions
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import json
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from PIL import Image, ImageOps
import gradio as gr
import os
print("TensorFlow", tf.__version__)


TensorFlow 2.20.0


/usr/local/lib/python3.12/dist-packages/gradio_client/documentation.py:106: UserWarning: Could not get documentation group for <class 'gradio.mix.Parallel'>: No known documentation group for module 'gradio.mix'
  warnings.warn(f"Could not get documentation group for {cls}: {exc}")
/usr/local/lib/python3.12/dist-packages/gradio_client/documentation.py:106: UserWarning: Could not get documentation group for <class 'gradio.mix.Series'>: No known documentation group for module 'gradio.mix'
  warnings.warn(f"Could not get documentation group for {cls}: {exc}")


In [ ]:
# Cell 3 — list available classes (run once to see names)
ds_info = tfds.builder("quickdraw_bitmap").info
all_names = ds_info.features["label"].names
print("Number of classes available:", len(all_names))
print("Sample class names (first 20):")
print(all_names[:200])  # show first 200 names; there are many


Number of classes available: 345
Sample class names (first 200):
['aircraft carrier', 'airplane', 'alarm clock', 'ambulance', 'angel', 'animal migration', 'ant', 'anvil', 'apple', 'arm', 'asparagus', 'axe', 'backpack', 'banana', 'bandage', 'barn', 'baseball bat', 'baseball', 'basket', 'basketball', 'bat', 'bathtub', 'beach', 'bear', 'beard', 'bed', 'bee', 'belt', 'bench', 'bicycle', 'binoculars', 'bird', 'birthday cake', 'blackberry', 'blueberry', 'book', 'boomerang', 'bottlecap', 'bowtie', 'bracelet', 'brain', 'bread', 'bridge', 'broccoli', 'broom', 'bucket', 'bulldozer', 'bus', 'bush', 'butterfly', 'cactus', 'cake', 'calculator', 'calendar', 'camel', 'camera', 'camouflage', 'campfire', 'candle', 'cannon', 'canoe', 'car', 'carrot', 'castle', 'cat', 'ceiling fan', 'cell phone', 'cello', 'chair', 'chandelier', 'church', 'circle', 'clarinet', 'clock', 'cloud', 'coffee cup', 'compass', 'computer', 'cookie', 'cooler', 'couch', 'cow', 'crab', 'crayon', 'crocodile', 'crown', 'cruise ship', '

In [ ]:
# Cell 4 — configure which classes to use and how many examples per class
CLASS_NAMES = [
    "cat","dog","house","tree","car","fish","duck","star","flower","bicycle",
    "airplane","ship","chair","cup","pizza","guitar","clock","key","apple","eye"
]
MAX_EXAMPLES_PER_CLASS = 200   # reduce to 50-100 for quicker runs (free Colab)
IMG_SIZE = 28


In [ ]:
# Cell 5 — build filtered dataset for chosen classes
import tensorflow_datasets as tfds
# Load the full quickdraw_bitmap train split (this may take a bit)
ds = tfds.load("quickdraw_bitmap", split="train", shuffle_files=True, as_supervised=True)

# Map numerical labels to names and vice versa
label_to_name = {i:name for i,name in enumerate(ds_info.features['label'].names)}
name_to_label = {name:i for i,name in label_to_name.items()}

# Get numerical labels for selected classes
selected_label_ids = [name_to_label[n] for n in CLASS_NAMES]
print("Selected original label ids:", selected_label_ids)

# Prepare a map for new sequential labels (0 to num_classes-1)
new_label_map = {original_id: new_id for new_id, original_id in enumerate(selected_label_ids)}

# List to store all collected image and label pairs
all_processed_examples = []

for class_name in CLASS_NAMES:
    original_class_id = name_to_label[class_name]
    new_class_id = new_label_map[original_class_id]

    # Filter the dataset for the current class and preprocess
    class_ds = ds.filter(lambda img, label: tf.equal(label, original_class_id)).map(
        lambda img, label: (tf.cast(tf.squeeze(img), tf.float32) / 255.0, tf.constant(new_class_id, dtype=tf.int32))
    ).take(MAX_EXAMPLES_PER_CLASS)

    # Collect these processed examples by iterating through the small, filtered dataset
    for img_data, label_data in tfds.as_numpy(class_ds):
        all_processed_examples.append((img_data, label_data))

# Separate images and labels into their respective lists
X_list = [item[0] for item in all_processed_examples]
y_list = [item[1] for item in all_processed_examples]

# Convert to numpy arrays
X = np.array(X_list, dtype='float32')
y = np.array(y_list, dtype='int32')

print(f"Collected {len(X)} examples in total across {len(CLASS_NAMES)} classes.")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

In [ ]:
# Cell 6 — create X,y arrays and train/test split
X_list = []
y_list = []
for idx, name in enumerate(CLASS_NAMES):
    arr = np.array(per_class_images[name], dtype='float32')
    # if any class has fewer than required, it still works but less data
    X_list.append(arr)
    y_list.append(np.full(len(arr), idx, dtype=np.int32))

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)
# reshape for channel
X = X.reshape((-1, IMG_SIZE, IMG_SIZE, 1))
# shuffle
perm = np.random.permutation(len(X))
X = X[perm]; y = y[perm]

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)

print("Train shape:", X_train.shape, "Val shape:", X_val.shape)
num_classes = len(CLASS_NAMES)
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)


In [ ]:
# Cell 7 — build model
def build_model(input_shape=(28,28,1), num_classes=num_classes):
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax'),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_model()
model.summary()


In [ ]:
# Cell 8 — train
EPOCHS = 12
BATCH = 128

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("model_quickdraw.h5", save_best_only=True, monitor='val_accuracy', mode='max')
]

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=callbacks
)


In [ ]:
# Cell 9 — save classes and model
model.save("model_quickdraw.h5")
with open("classes_quickdraw.json","w") as f:
    json.dump(CLASS_NAMES, f)
print("Saved model_quickdraw.h5 and classes_quickdraw.json")


In [ ]:
# Cell 10 — evaluate on validation
loss, acc = model.evaluate(X_val, y_val_cat, verbose=0)
print(f"Validation accuracy: {acc:.4f}")


In [ ]:
# Cell 11 — Gradio inference function
from PIL import Image
import numpy as np
import io

# load model and classes (ensure they exist)
clf = tf.keras.models.load_model("model_quickdraw.h5")
with open("classes_quickdraw.json","r") as f:
    class_names = json.load(f)

def preprocess_pil(img: Image, size=(28,28)):
    # img: PIL RGBA or RGB
    img = img.convert("L")           # grayscale
    img = ImageOps.invert(img)      # invert: user draws black on white -> invert to white background?
    # NOTE: quickdraw images are white background black strokes (0..255), training used 0..1 as is
    # We invert because Gradio canvas background is white and strokes are black -> after convert L it's 255 background, 0 strokes
    # QuickDraw bitmaps use 0 for background, 255 for stroke? If predictions look swapped remove invert().
    img = img.resize(size, Image.ANTIALIAS)
    arr = np.array(img).astype('float32') / 255.0
    arr = arr.reshape((1, size[0], size[1], 1))
    return arr

def predict_gradio(pil_img):
    # pil_img is a PIL image from Gradio sketchpad
    arr = preprocess_pil(pil_img, size=(IMG_SIZE, IMG_SIZE))
    preds = clf.predict(arr)[0]
    top_idx = preds.argsort()[-3:][::-1]
    result = [(class_names[i], float(preds[i])) for i in top_idx]
    # Format nicely: return label strings and confidences
    labels = [f"{name} ({prob:.3f})" for name, prob in result]
    return {labels[0]: preds[top_idx[0]], labels[1]: preds[top_idx[1]], labels[2]: preds[top_idx[2]]}

# Build the Gradio interface
title = "🎨 Guess What I Drew (Colab demo)"
description = "Draw something (use the pencil). The model will return top-3 guesses. If predictions seem inverted, try toggling invert line in preprocess."

with gr.Blocks() as demo:
    gr.Markdown(f"# {title}\n\n{description}")
    with gr.Row():
        sketch = gr.Sketchpad(label="Draw here", shape=(280,280))
        output = gr.Label(num_top_classes=3, label="Top 3 guesses")
    btn = gr.Button("Predict")
    btn.click(fn=predict_gradio, inputs=sketch, outputs=output)

# Launch with share=True so you get a public URL
demo.launch(share=True, debug=True)
